In [ ]:
import osimport sysprint("🚀 Auto Clipper By MANG OTHIB - Smart Setup")print("=" * 60)# STEP 1: Install Dependencies (Smart Check)print("\n📦 Lagi nginstal dulu nih, lo ngopi dulu gih...")print("   (Biasanya 2-3 menit sih, sabar ya)\n")# Check if already installeddeps_installed = Falsetry:    import dotenv, pytubefix, moviepy, faster_whisper, torch, google.generativeai, cv2, numpy, PIL, tqdm    print("   ✅ Wah, udah keinstall semua tuh!")    deps_installed = Trueexcept ImportError:    print("   ⏳ Oke, mulai install sekarang...")    deps_installed = Falseif not deps_installed:    !pip install -q python-dotenv pytubefix moviepy google-generativeai pillow yt-dlp    !pip install -q faster-whisper    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118    !pip install -q opencv-python    !pip install -q tqdm    print("   ✅ Oke, semua dependency udah keinstall!")# STEP 2: Verify OpenCV (for face tracking)print("\n🎯 Cek OpenCV buat face tracking dulu...")# OpenCV should already be installed, just verifytry:    import cv2    # Test Haar Cascade availability    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'    test_cascade = cv2.CascadeClassifier(cascade_path)    if test_cascade.empty():        raise RuntimeError("Haar Cascade not found")    print("   ✅ OpenCV siap nih, face detection oke!")    print(f"   📌 OpenCV Version: {cv2.__version__}")    print("   💡 Pakai Haar Cascade (enggak ada masalah MediaPipe!)")except Exception as e:    print(f"   ⚠️  OpenCV error nih: {e}")    print("   🔄 Restart runtime dulu, terus jalanin cell ini lagi ya!")    raise# STEP 3: Setup Directoriesprint("\n📂 Bikin folder output sama temp dulu...")os.makedirs("output", exist_ok=True)os.makedirs("temp", exist_ok=True)print("   ✅ Folder udah siap nih!")# STEP 4: Setup Gemini API Keyprint("\n🔑 Masukin API key Gemini lo dulu...")from pathlib import Pathfrom getpass import getpass# Check if API key already setenv_file = Path(".env")if env_file.exists():    from dotenv import load_dotenv    load_dotenv()    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')    if GEMINI_API_KEY and "YOUR_API_KEY_HERE" not in GEMINI_API_KEY:        print("   ✅ API key lo udah ada nih!")    else:        print("   ⚠️  API key lo belum ada atau belum valid.")        print("   💡 Dapatin API key gratis di: https://aistudio.google.com/app/apikey")        api_key = getpass("   🔑 Masukin API key Gemini lo: ")                if api_key:            with open(".env", "w") as f:                f.write(f"GEMINI_API_KEY={api_key}\n")            GEMINI_API_KEY = api_key            print("   ✅ API key lo udah disimpan!")        else:            print("   ⚠️  Lo belum masukin API key. Nanti masukin manual ya.")            GEMINI_API_KEY = ""else:    print("   ⚠️  API key lo belum ada nih.")    print("   💡 Dapatin API key gratis di: https://aistudio.google.com/app/apikey")    api_key = getpass("   🔑 Masukin API key Gemini lo: ")        if api_key:        with open(".env", "w") as f:            f.write(f"GEMINI_API_KEY={api_key}\n")        GEMINI_API_KEY = api_key        print("   ✅ API key lo udah disimpan!")    else:        print("   ⚠️  Lo belum masukin API key. Nanti masukin manual ya.")        GEMINI_API_KEY = ""# STEP 5: Check GPUprint("\n🖥️  Cek GPU lo dulu...")try:    import torch    if torch.cuda.is_available():        gpu_name = torch.cuda.get_device_name(0)        print(f"   ✅ GPU lo ada nih: {gpu_name}")        print(f"   📌 CUDA Version: {torch.version.cuda}")        print(f"   📌 PyTorch Version: {torch.__version__}")    else:        print("   ⚠️  GPU lo belum aktif. Aktifin dulu: Runtime → Change runtime type → T4 GPU")except Exception as e:    print(f"   ⚠️  Cek GPU gagal: {e}")# DONE!print("\n" + "=" * 60)print("✅ Setup selesai! Siap bikin viral clips nih!")print("=" * 60)print("\n✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print("\n📌 Lanjutin ke cell berikutnya buat proses videonya!\n")

In [ ]:
import osimport tempfileimport randomimport jsonimport cv2import numpy as npfrom pathlib import Pathfrom urllib.parse import urlparse, parse_qsfrom PIL import Image, ImageDraw, ImageFontfrom pytubefix import YouTubefrom pytubefix.exceptions import PytubeFixErrorfrom faster_whisper import WhisperModelimport torchimport google.generativeai as genaifrom moviepy.editor import VideoFileClip, ImageClip, CompositeVideoClipfrom tqdm import tqdmprint("📚 Lagi define semua class sama function nih...")# ConfigurationOUTPUT_DIR = Path('./output')TEMP_DIR = Path('./temp')WHISPER_MODEL = 'medium'YOUTUBE_USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'# Caption StylesCAPTION_STYLES = {    'clean_white': {        'text_color': (255, 255, 255, 255),        'font_type': 'bold',        'name': 'Clean White',        'description': 'Simple white text, clean and readable'    },    'bright_yellow': {        'text_color': (255, 255, 0, 255),        'font_type': 'bold',        'name': 'Bright Yellow',        'description': 'Bold yellow text, eye-catching and energetic'    },    'neon_cyan': {        'text_color': (0, 255, 255, 255),        'font_type': 'regular',        'name': 'Neon Cyan',        'description': 'Cool cyan/turquoise, modern and fresh'    },    'hot_pink': {        'text_color': (255, 20, 147, 255),        'font_type': 'bold',        'name': 'Hot Pink',        'description': 'Vibrant pink, fun and attention-grabbing'    },    'lime_green': {        'text_color': (50, 205, 50, 255),        'font_type': 'regular',        'name': 'Lime Green',        'description': 'Bright green, energetic and positive'    },    'orange_fire': {        'text_color': (255, 165, 0, 255),        'font_type': 'bold',        'name': 'Orange Fire',        'description': 'Warm orange, dynamic and exciting'    },    'electric_blue': {        'text_color': (30, 144, 255, 255),        'font_type': 'regular',        'name': 'Electric Blue',        'description': 'Bright blue, professional and trustworthy'    },    'purple_pop': {        'text_color': (138, 43, 226, 255),        'font_type': 'bold',        'name': 'Purple Pop',        'description': 'Bold purple, creative and unique'    }}HIGHLIGHT_KEYWORDS = [    'amazing', 'incredible', 'secret', 'important', 'shocking', 'exclusive',    'never', 'always', 'only', 'must', 'can\'t', 'won\'t', 'best', 'worst',    'first', 'last', 'biggest', 'smallest', 'most', 'least', 'why', 'how',    'what', 'when', 'where', 'money', 'free', 'easy', 'hard', 'truth']print("   ✅ Config udah loaded!")# Helper Functionsdef cleanup_temp_files():    """Removes all files from the temporary directory."""    print("🧹 Lagi bersih-bersih file temporary...")    try:        for item in TEMP_DIR.iterdir():            if item.is_file():                item.unlink()        print("✅ Cleanup selesai.")    except Exception as e:        print(f"⚠️  Enggak bisa bersih-bersih semua file temporary: {e}")def generate_random_clips(duration, num_clips, min_duration, max_duration):    """Generates random clip start and end times as a fallback."""    clips = []        # Make sure we have valid parameters    if duration < min_duration:        print(f"⚠️  Video duration ({duration}s) is shorter than min_duration ({min_duration}s)")        return clips        # Adjust max_duration if video is too short    actual_max_dur = min(max_duration, duration - 1)    actual_min_dur = min(min_duration, actual_max_dur)        for i in range(num_clips):        # Generate random duration between min and max        clip_duration = random.uniform(actual_min_dur, actual_max_dur)                # Make sure there's space for this clip        if duration - clip_duration <= 0:            continue                # Random start position        start = random.uniform(0, duration - clip_duration)        end = start + clip_duration                clips.append({            'start': start,            'end': end,            'title': f'Random clip {i+1}',            'virality_score': 30,            'hook_type': 'general',            'duration': clip_duration        })        return clipsprint("   ✅ Helper functions udah siap!")# YouTubeDownloader Classclass YouTubeDownloader:    """Handles the downloading of YouTube videos."""        def __init__(self, temp_dir=TEMP_DIR):        self.temp_dir = temp_dir        self.progress_bar = None            def _sanitize_filename(self, filename):        """Sanitizes a filename by removing invalid characters."""        if not filename:            return "unknown_title"        invalid_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*']        for char in invalid_chars:            filename = filename.replace(char, '_')        if len(filename) > 100:            filename = filename[:97] + '...'        return filename    @staticmethod    def get_video_id(url):        """Extracts the video ID from a YouTube URL."""        if 'youtu.be' in url:            return url.split('/')[-1].split('?')[0]        if 'youtube.com' in url:            return parse_qs(urlparse(url).query).get('v', [None])[0]        return None        def _on_progress(self, stream, chunk, bytes_remaining):        """Callback function untuk progress bar download."""        total_size = stream.filesize                if total_size is None or total_size == 0:            return                bytes_downloaded = total_size - bytes_remaining        percent = (bytes_downloaded / total_size) * 100                if self.progress_bar is None:            self.progress_bar = tqdm(                total=total_size,                unit='B',                unit_scale=True,                unit_divisor=1024,                desc="📥 Download",                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] {postfix}',                initial=0            )                # Update progress bar        current_pos = total_size - bytes_remaining        self.progress_bar.n = current_pos        self.progress_bar.refresh()        self.progress_bar.set_postfix({'Progress': f'{percent:.1f}%'})                # Jika sudah selesai        if bytes_remaining == 0:            self.progress_bar.n = total_size            self.progress_bar.refresh()        def _close_progress_bar(self):        """Menutup progress bar."""        if self.progress_bar is not None:            self.progress_bar.close()            self.progress_bar = None    def download(self, url):        """Downloads a YouTube video from the given URL."""        print(f"🔗 Lagi proses URL YouTube lo: {url}")        video_id = self.get_video_id(url)        if not video_id:            raise ValueError("URL YouTube lo salah nih.")        print(f"✅ Video ID lo: {video_id}")        os.makedirs(self.temp_dir, exist_ok=True)        output_path = os.path.abspath(str(self.temp_dir))        output_filename = f"{video_id}.mp4"                try:            print(f"⏳ Lagi ambil info video lo...")            # Setup progress callback            yt = YouTube(url, on_progress_callback=self._on_progress)            print(f"✅ Konek ke YouTube API berhasil!")                        title = self._sanitize_filename(yt.title)            duration = yt.length                        print(f"Judul video: {title}")            print(f"Durasi video: {duration} detik ({duration//60}m {duration%60}s)")            print(f"⏳ Lagi cari kualitas HD terbaik...")                        # Priority 1: Try adaptive streams FIRST (for highest resolution: 4K, 1440p, 1080p)            # Adaptive streams are required for resolutions above 720p            print("🔍 Lagi cari stream kualitas tertinggi...")                        # Get all available video streams sorted by resolution            all_video_streams = (                yt.streams                .filter(adaptive=True, only_video=True)                .order_by('resolution')                .desc()            )                        # Show available resolutions for debugging            available_resolutions = [s.resolution for s in all_video_streams if s.resolution]            if available_resolutions:                unique_res = sorted(set(available_resolutions), key=lambda x: int(x.replace('p', '')) if x.replace('p', '').isdigit() else 0, reverse=True)                print(f"📊 Resolusi yang tersedia: {', '.join(unique_res)}")                        # Get best audio stream            audio_stream = (                yt.streams                .filter(adaptive=True, only_audio=True)                .order_by('abr')                .desc()                .first()            )                        # Try to find best video stream (prefer mp4, but accept webm if higher quality)            video_stream = None            preferred_resolutions = ['4320p', '2160p', '1440p', '1080p', '720p', '480p', '360p']                        # Strategy: Get highest resolution available, prefer mp4 but use webm if higher res            best_mp4 = None            best_webm = None                        # Find best mp4 stream            for res in preferred_resolutions:                candidate = all_video_streams.filter(file_extension='mp4', res=res).first()                if candidate:                    best_mp4 = candidate                    break                        # Find best webm stream            for res in preferred_resolutions:                candidate = all_video_streams.filter(file_extension='webm', res=res).first()                if candidate:                    best_webm = candidate                    break                        # Choose best: prefer higher resolution, mp4 if same resolution            if best_mp4 and best_webm:                # Compare resolutions (extract number)                mp4_res_num = int(best_mp4.resolution.replace('p', '')) if best_mp4.resolution else 0                webm_res_num = int(best_webm.resolution.replace('p', '')) if best_webm.resolution else 0                                if webm_res_num > mp4_res_num:                    video_stream = best_webm                    print(f"   📹 Pake WebM buat resolusi lebih tinggi: {best_webm.resolution}")                else:                    video_stream = best_mp4                    print(f"   📹 Pake MP4: {best_mp4.resolution}")            elif best_mp4:                video_stream = best_mp4                print(f"   📹 Pake MP4: {best_mp4.resolution}")            elif best_webm:                video_stream = best_webm                print(f"   📹 Pake WebM: {best_webm.resolution}")            else:                # Fallback: get highest available                video_stream = all_video_streams.first()                if video_stream:                    print(f"   📹 Pake stream yang tersedia: {video_stream.resolution}")                        # Combine adaptive streams if both available            if video_stream and audio_stream:                print(f"✅ Ketemu video kualitas terbaik: {video_stream.resolution} ({video_stream.mime_type})")                print(f"✅ Ketemu audio terbaik: {audio_stream.abr}")                print(f"⏳ Lagi download video stream...")                                # Download video dengan progress bar                self.progress_bar = None                video_path = video_stream.download(output_path=output_path, filename=f"video_{video_id}.{video_stream.subtype}")                self._close_progress_bar()                                print(f"⏳ Lagi download audio stream...")                # Download audio dengan progress bar                self.progress_bar = None                audio_path = audio_stream.download(output_path=output_path, filename=f"audio_{video_id}.{audio_stream.subtype}")                self._close_progress_bar()                                # Combine using ffmpeg                print(f"⏳ Lagi gabungin video sama audio buat kualitas maksimal...")                final_path = os.path.join(output_path, output_filename)                import subprocess                subprocess.run([                    'ffmpeg', '-i', video_path, '-i', audio_path,                    '-c:v', 'copy', '-c:a', 'aac', '-b:a', '320k',                    final_path, '-y'                ], check=True, capture_output=True)                                # Clean up temp files                os.remove(video_path)                os.remove(audio_path)                                video_path = final_path  # Use combined file                stream = None  # Mark as combined            else:                # Fallback to progressive streams if adaptive not available                print("⚠️ Adaptive stream enggak ada, coba progressive stream...")                stream = (                    yt.streams                    .filter(progressive=True, file_extension='mp4')                    .order_by('resolution')                    .desc()                    .first()                )                        # Download stream if not already combined from adaptive streams            if stream:                print(f"Stream yang dipilih: {stream.resolution}, {stream.mime_type}")                print(f"⏳ Mulai download...")                # Download dengan progress bar                self.progress_bar = None                video_path = stream.download(output_path=output_path, filename=output_filename)                self._close_progress_bar()            elif 'video_path' not in locals():                # If adaptive streams were combined, video_path should already be set                # Otherwise, check if file exists                potential_path = os.path.join(output_path, output_filename)                if os.path.exists(potential_path):                    video_path = potential_path                else:                    raise ValueError("No suitable video stream found")                        # Verify downloaded file            if not os.path.exists(video_path) or os.path.getsize(video_path) == 0:                raise FileNotFoundError(f"Downloaded file is missing or empty")                        file_size_mb = os.path.getsize(video_path) / (1024 * 1024)                        # Get video resolution for info            try:                from moviepy.editor import VideoFileClip                temp_clip = VideoFileClip(str(video_path))                width, height = temp_clip.size                temp_clip.close()                                # Determine quality level                if height >= 2160:                    quality = "4K UHD"                elif height >= 1440:                    quality = "1440p QHD"                elif height >= 1080:                    quality = "1080p Full HD"                elif height >= 720:                    quality = "720p HD"                elif height >= 480:                    quality = "480p SD"                else:                    quality = "360p or lower"                                print(f"✅ Download selesai!")                print(f"   Resolusi: {width}x{height} ({quality})")                print(f"   Ukuran file: {file_size_mb:.2f} MB")                                # Warning if resolution is low                if height < 720:                    print(f"   ⚠️  WARNING: Resolusi videonya rendah nih. Mungkin enggak ada versi HD di YouTube.")                    print(f"   💡 Coba video lain atau cek apakah ada versi HD-nya.")            except Exception as e:                print(f"✅ Download selesai! Ukuran file: {file_size_mb:.2f} MB")                print(f"   ⚠️  Enggak bisa detect resolusi: {e}")                        # Pastikan progress bar ditutup            self._close_progress_bar()            return Path(video_path), title, duration                    except Exception as e:            # Pastikan progress bar ditutup meskipun ada error            self._close_progress_bar()            print(f"❌ Error: {str(e)}")            raise Exception(f"Download video gagal: {str(e)}")print("   ✅ YouTubeDownloader class udah siap!")# WhisperSingleton Classclass WhisperSingleton:    """A singleton class for transcribing audio using faster-whisper."""        _instance = None    _model = None    def __new__(cls):        if cls._instance is None:            cls._instance = super().__new__(cls)            cls._instance._load_model()        return cls._instance    def _load_model(self):        """Loads the faster-whisper model with optimized settings."""        if self._model is None:            print(f"Lagi load faster-whisper model ({WHISPER_MODEL})... (cuma sekali nih)")            try:                compute_type = "float16" if torch.cuda.is_available() else "int8"                device = "cuda" if torch.cuda.is_available() else "cpu"                os.environ["OMP_NUM_THREADS"] = "1"                                self._model = WhisperModel(                    WHISPER_MODEL,                    device=device,                    compute_type=compute_type,                    cpu_threads=4,                    num_workers=2                )                print(f"✅ faster-whisper model udah loaded di: {device} dengan {compute_type} precision")            except Exception as e:                print(f"Error load model, fallback ke CPU: {e}")                self._model = WhisperModel(                    WHISPER_MODEL,                    device="cpu",                    compute_type="int8",                    cpu_threads=2,                    num_workers=1                )    def transcribe(self, video_path):        """Transcribes the audio from a video file with automatic language detection."""        print("🎵 Lagi transcribe video lo...")        try:            print("⏳ Mulai proses audio dengan auto detect bahasa...")                        # First pass: Auto-detect language            print("🔍 Lagi detect bahasa videonya...")            segments, info = self._model.transcribe(                str(video_path),                 word_timestamps=True,                vad_filter=True,                vad_parameters={"min_silence_duration_ms": 500},                language=None,  # Auto-detect language                beam_size=1,                best_of=1,                temperature=0            )                        # Get detected language            detected_language = info.language if hasattr(info, 'language') else None            language_probability = info.language_probability if hasattr(info, 'language_probability') else None                        # Map language codes to full names            language_names = {                'id': 'Indonesian',                'en': 'English',                'ms': 'Malay',                'th': 'Thai',                'vi': 'Vietnamese',                'zh': 'Chinese',                'ja': 'Japanese',                'ko': 'Korean'            }                        language_name = language_names.get(detected_language, detected_language or 'Unknown')                        if detected_language:                confidence = f"{language_probability*100:.1f}%" if language_probability else "N/A"                print(f"✅ Bahasa yang terdeteksi: {language_name} ({detected_language}) - Confidence: {confidence}")                                # Re-transcribe with detected language for better accuracy                if detected_language in ['id', 'en']:                    print(f"🔄 Re-transcribe pake {language_name} buat hasil lebih akurat...")                    segments, info = self._model.transcribe(                        str(video_path),                         word_timestamps=True,                        vad_filter=True,                        vad_parameters={"min_silence_duration_ms": 500},                        language=detected_language,  # Use detected language                        beam_size=1,                        best_of=1,                        temperature=0                    )            else:                print("⚠️  Enggak bisa detect bahasa, pake auto-detect mode aja")                language_name = "Auto-detect"                        print("✅ Proses audio selesai, lagi extract kata-katanya...")                        words = []            segments_list = []            full_text = ""            segment_count = 0                        for segment in segments:                segment_count += 1                if segment_count % 10 == 0:                    print(f"⏳ Processed {segment_count} segments...")                                    segments_list.append({                    'id': segment.id,                    'start': segment.start,                    'end': segment.end,                    'text': segment.text                })                full_text += segment.text + " "                                if hasattr(segment, 'words') and segment.words:                    for word_info in segment.words:                        word = word_info.word.strip().upper()                        if word:                            words.append({                                'word': word,                                 'start': word_info.start,                                 'end': word_info.end                            })                        print(f"✅ Transcribe selesai! Ketemu {len(words)} kata dalam {len(segments_list)} segment")            print(f"🌐 Bahasa: {language_name} ({detected_language or 'auto'})")                        # Store detected language for later use            self.detected_language = detected_language            self.language_name = language_name                        return words, full_text, segments_list, detected_language        except Exception as e:            print(f"❌ Transcribe gagal: {e}")            return [], "", []print("   ✅ WhisperSingleton class udah siap!")# GeminiSelector Classclass GeminiSelector:    """Uses the Gemini AI model to select the most viral clips."""        def __init__(self, api_key):        self.api_key = api_key        genai.configure(api_key=self.api_key)                # Try different models (gemini-2.5-flash is latest and best, with fallbacks)        model_names = ['gemini-2.5-flash', 'gemini-1.5-flash', 'gemini-1.5-pro', 'gemini-pro']        self.model = None        self.model_name = None                for model_name in model_names:            try:                self.model = genai.GenerativeModel(model_name)                self.model_name = model_name                print(f"✅ Pake Gemini model: {model_name}")                break            except Exception as e:                print(f"⚠️  Model {model_name} enggak tersedia: {str(e)[:100]}")                continue                if self.model is None:            raise ValueError("No compatible Gemini model found. Please check your API key.")    def select_clips(self, segments, video_duration, n, min_dur, max_dur, sentiment=None, language='id'):        """Selects the most viral clips using AI, optionally filtered by sentiment."""        segments_text = []        for seg in segments:            segments_text.append(f"[{seg['start']:.1f}s-{seg['end']:.1f}s]: {seg['text']}")                transcript_with_timestamps = "\n".join(segments_text)                # Build sentiment instruction based on language        sentiment_instruction = ""        if language == 'en':            # English sentiment examples            sentiment_examples = {                "lucu": "Find moments that make people laugh, jokes, funny situations, funny expressions",                "motivasi": "Find words that inspire, encourage action, inspirational quotes",                "edukasi": "Find useful explanations, interesting facts, tutorials, tips & tricks",                "dramatis": "Find intense moments, turning points, conflicts, surprising reveals",                "emosional": "Find touching moments, personal stories, struggles, achievements",                "inspirasi": "Find success stories, overcoming obstacles, transformation journeys",                "kontroversial": "Find strong opinions, debates, hot takes, unique perspectives",                "tutorial": "Find clear steps, how-to, demonstrations, practical explanations",                "storytelling": "Find engaging narratives, interesting plots, character development"            }                        if sentiment:                example = sentiment_examples.get(sentiment.lower(), "Find moments that best match this sentiment")                sentiment_instruction = f"""🎭 TARGET SENTIMENT: {sentiment.upper()}   FOCUS: {example}   PRIORITY: Select ONLY clips that STRONGLY reflect the sentiment '{sentiment}'   DO NOT select clips that are not relevant to this sentiment!"""                        prompt = f"""You are an expert at creating viral short-form content. Select the {n} BEST viral clips from this transcript.{sentiment_instruction}IMPORTANT RULES:1. Each clip MUST be {min_dur}-{max_dur} seconds long2. Only select complete thoughts - do not cut mid-sentence3. Focus on: engaging hooks, revelations, advice, stories, moments that match the sentiment4. Use EXACT timestamps from the provided transcriptVIDEO DURATION: {video_duration} secondsTRANSCRIPT WITH TIMESTAMPS:{transcript_with_timestamps}Return ONLY valid JSON:{{  "clips": [    {{      "start": 34.5,      "end": 67.2,      "title": "Engaging clip title",      "virality_score": 85,      "hook_type": "story_reveal"    }}  ]}}"""        else:            # Indonesian sentiment examples (default)            sentiment_examples = {                "lucu": "Cari momen yang membuat orang tertawa, jokes, situasi kocak, ekspresi lucu",                "motivasi": "Cari kata-kata yang membangkitkan semangat, dorongan untuk action, quote inspiratif",                "edukasi": "Cari penjelasan yang bermanfaat, fakta menarik, tutorial, tips & tricks",                "dramatis": "Cari momen intens, turning point, konflik, reveal yang mengejutkan",                "emosional": "Cari momen yang menyentuh hati, cerita pribadi, struggles, achievement",                "inspirasi": "Cari success stories, overcome obstacles, transformation journey",                "kontroversial": "Cari opini kuat, perdebatan, hot takes, perspektif unik",                "tutorial": "Cari langkah-langkah jelas, how-to, demonstrasi, penjelasan praktis",                "storytelling": "Cari narasi yang engaging, plot menarik, character development"            }                        if sentiment:                example = sentiment_examples.get(sentiment.lower(), "Cari momen yang paling sesuai dengan sentimen ini")                sentiment_instruction = f"""🎭 SENTIMEN TARGET: {sentiment.upper()}   FOKUS: {example}   PRIORITAS: Pilih HANYA klip yang KUAT mencerminkan sentimen '{sentiment}'   JANGAN pilih klip yang tidak relevan dengan sentimen ini!"""                        prompt = f"""Kamu adalah ahli dalam membuat konten viral bentuk pendek (short-form content). Pilih {n} klip TERBAIK yang paling viral dari transkrip ini.{sentiment_instruction}ATURAN PENTING:1. Setiap klip HARUS berdurasi {min_dur}-{max_dur} detik2. Hanya pilih pemikiran yang lengkap - jangan potong di tengah kalimat3. Fokus pada: hooks menarik, revelasi, nasihat, cerita, momen yang sesuai sentimen4. Gunakan timestamp PERSIS dari transkrip yang diberikanDURASI VIDEO: {video_duration} detikTRANSKRIP DENGAN TIMESTAMP:{transcript_with_timestamps}Kembalikan HANYA JSON yang valid:{{  "clips": [    {{      "start": 34.5,      "end": 67.2,      "title": "Judul klip yang menarik",      "virality_score": 85,      "hook_type": "story_reveal"    }}  ]}}"""                try:            print("🤖 AI lagi analisis transcript lo...")            response = self.model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})            data = json.loads(response.text)            validated_clips = []                        for clip_data in data.get('clips', []):                start = clip_data.get('start')                end = clip_data.get('end')                                if start is None or end is None:                    continue                start, end = float(start), float(end)                duration = end - start                                if duration > max_dur:                    end = start + max_dur                    duration = max_dur                                    if min_dur <= duration <= max_dur and start < end and end <= video_duration:                    validated_clips.append({                        'start': start,                        'end': end,                        'title': clip_data.get('title', 'Untitled'),                        'virality_score': clip_data.get('virality_score', 0),                        'hook_type': clip_data.get('hook_type', 'general'),                        'duration': duration                    })            if not validated_clips:                raise ValueError("AI enggak return clips yang valid")            validated_clips.sort(key=lambda x: x['virality_score'], reverse=True)            print(f"✅ AI udah pilih {len(validated_clips)} viral clips")            return validated_clips[:n]                    except Exception as e:            print(f"❌ AI selection gagal: {e}. Pake fallback")            return self._fallback_selection(segments, video_duration, n, min_dur, max_dur, sentiment=sentiment)    def _fallback_selection(self, segments, video_duration, n, min_dur, max_dur, sentiment=None):        """Fallback random selection (sentiment cannot be applied in fallback mode)."""        if sentiment:            print(f"⚠️  Fallback mode: Sentimen '{sentiment}' enggak bisa diterapkan (pilih acak aja)")        clips = []        used_segments = set()                for i in range(n):            available = [seg for j, seg in enumerate(segments) if j not in used_segments]            if not available:                break                            segment = random.choice(available)            seg_idx = segments.index(segment)            used_segments.add(seg_idx)                        start = segment['start']                        # Calculate proper duration            # Try to extend to next segments if needed            end = segment['end']            duration = end - start                        # If duration too short, extend to next segments            current_idx = seg_idx            while duration < min_dur and current_idx + 1 < len(segments):                current_idx += 1                next_seg = segments[current_idx]                end = next_seg['end']                duration = end - start                used_segments.add(current_idx)                                # Stop if we reached max_dur                if duration >= max_dur:                    end = start + max_dur                    duration = max_dur                    break                        # If still too short, just use max_dur from start            if duration < min_dur:                duration = min(max_dur, video_duration - start)                end = start + duration                        # Make sure we don't exceed max_dur            if duration > max_dur:                duration = max_dur                end = start + duration                        # Make sure we don't exceed video duration            if end > video_duration:                end = video_duration                duration = end - start                        # Only add if duration is valid            if duration >= min_dur:                clips.append({                    'start': start,                    'end': end,                    'title': f'Clip {i+1}',                    'virality_score': 50,                    'hook_type': 'general',                    'duration': duration                })                return clipsprint("   ✅ GeminiSelector class udah siap!")# SEO Content Generator Classclass SEOContentGenerator:    """Generates SEO-optimized content (title, description, hashtags, tags) for video clips."""        def __init__(self, api_key):        self.api_key = api_key        genai.configure(api_key=self.api_key)                # Try different models        model_names = ['gemini-2.5-flash', 'gemini-1.5-flash', 'gemini-1.5-pro', 'gemini-pro']        self.model = None        self.model_name = None                for model_name in model_names:            try:                self.model = genai.GenerativeModel(model_name)                self.model_name = model_name                break            except Exception as e:                continue                if self.model is None:            raise ValueError("No compatible Gemini model found for SEO generation.")    def generate_seo_content(self, clip_title, clip_transcript, language='id', platform='youtube'):        """Generates SEO-optimized content for a video clip."""        try:            if language == 'en':                prompt = f"""You are an expert SEO content creator for {platform.upper()} Shorts. Generate SEO-optimized content for this video clip.CLIP TITLE: {clip_title}CLIP CONTENT: {clip_transcript[:500]}...Generate SEO-optimized content in JSON format:{{  "title": "Engaging, keyword-rich title (max 60 chars, include main keyword)",  "description": "SEO-optimized description (150-200 words, include keywords naturally, call-to-action)",  "hashtags": ["#hashtag1", "#hashtag2", ...] (10-15 relevant hashtags),  "tags": ["tag1", "tag2", ...] (15-20 relevant tags, comma-separated keywords)}}RULES:- Title: Catchy, includes main keyword, under 60 characters- Description: SEO-friendly, includes keywords naturally, has CTA, 150-200 words- Hashtags: Mix of trending and niche hashtags (10-15 total)- Tags: Relevant keywords for SEO (15-20 tags)- All content must be engaging and encourage clicks/views- Optimize for {platform.upper()} Shorts algorithmReturn ONLY valid JSON:"""            else:                prompt = f"""Kamu adalah ahli SEO content creator untuk {platform.upper()} Shorts. Generate konten SEO-optimized untuk klip video ini.JUDUL KLIP: {clip_title}ISI KLIP: {clip_transcript[:500]}...Generate konten SEO-optimized dalam format JSON:{{  "title": "Judul menarik, kaya keyword (max 60 karakter, include keyword utama)",  "description": "Deskripsi SEO-optimized (150-200 kata, include keywords secara natural, call-to-action)",  "hashtags": ["#hashtag1", "#hashtag2", ...] (10-15 hashtag relevan),  "tags": ["tag1", "tag2", ...] (15-20 tag relevan, keyword dipisah koma)}}ATURAN:- Title: Menarik, include keyword utama, maksimal 60 karakter- Description: SEO-friendly, include keywords natural, ada CTA, 150-200 kata- Hashtags: Mix trending dan niche hashtags (10-15 total)- Tags: Keyword relevan untuk SEO (15-20 tags)- Semua konten harus engaging dan encourage clicks/views- Optimize untuk algoritma {platform.upper()} ShortsKembalikan HANYA JSON yang valid:"""                        response = self.model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})            data = json.loads(response.text)                        return {                'title': data.get('title', clip_title),                'description': data.get('description', ''),                'hashtags': data.get('hashtags', []),                'tags': data.get('tags', [])            }                    except Exception as e:            print(f"⚠️  Generate SEO gagal: {e}")            # Fallback: Generate basic SEO content            return {                'title': clip_title[:60] if len(clip_title) > 60 else clip_title,                'description': f"{clip_title}. Tonton viral clip ini! 🔥",                'hashtags': ['#viral', '#shorts', '#trending'],                'tags': ['viral', 'shorts', 'trending']            }print("   ✅ SEOContentGenerator class udah siap!")# FaceTracker Classclass FaceTracker:    """Tracks faces and crops video using OpenCV Haar Cascade."""        def __init__(self):        cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'        self.face_cascade = cv2.CascadeClassifier(cascade_path)                if self.face_cascade.empty():            raise RuntimeError("Gagal load face cascade")                self.face_cache = {}        print("🎯 Face tracking dengan OpenCV udah siap")    def detect_faces_in_frame(self, frame, frame_time=None):        """Detects faces in a single frame."""        if frame_time is not None and frame_time in self.face_cache:            return self.face_cache[frame_time]                    try:            h, w, _ = frame.shape            scale = 0.5            small_frame = cv2.resize(frame, (int(w*scale), int(h*scale)))            gray_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)                        detected_faces = self.face_cascade.detectMultiScale(                gray_frame,                scaleFactor=1.1,                minNeighbors=5,                minSize=(30, 30)            )            faces = []            for (x, y, width, height) in detected_faces:                x = int(x / scale)                y = int(y / scale)                width = int(width / scale)                height = int(height / scale)                center_x = x + width // 2                center_y = y + height // 2                area = width * height                confidence = min(1.0, area / (w * h * 0.5))                faces.append({                    'center_x': center_x,                    'center_y': center_y,                    'width': width,                    'height': height,                    'confidence': confidence,                    'area': area                })            result = sorted(faces, key=lambda f: f['confidence'] * f['area'], reverse=True)                        if frame_time is not None:                self.face_cache[frame_time] = result                            return result        except Exception as e:            print(f"    ⚠️ Error detect wajah: {e}")            return []    def smooth_trajectory(self, positions, window_size=5):        """Smoothes a trajectory using moving average."""        if len(positions) <= window_size:            return positions        smoothed = []        for i in range(len(positions)):            start_idx = max(0, i - window_size // 2)            end_idx = min(len(positions), i + window_size // 2 + 1)            window = positions[start_idx:end_idx]            avg_x = sum(pos[0] for pos in window) / len(window)            avg_y = sum(pos[1] for pos in window) / len(window)            smoothed.append((avg_x, avg_y))        return smoothed    def track_and_crop(self, clip):        """Tracks faces and crops video to 9:16 (YouTube Shorts format)."""        width, height = clip.size        target_width = int(height * 9 / 16)        if target_width % 2 != 0:            target_width -= 1                # Always crop to 9:16 for YouTube Shorts        if width <= target_width:            print("    ⏩ Video lo udah dalam format 9:16")            return clip        print("    🎯 Lagi analisis frame-frame...")        self.face_cache = {}                face_positions = []        num_samples = min(6, max(3, int(clip.duration / 3)))                print(f"    ⏳ Lagi analisis {num_samples} frame...")        sample_times = np.linspace(0, clip.duration, num_samples)        for i, t in enumerate(sample_times):            try:                frame = clip.get_frame(t)                faces = self.detect_faces_in_frame(frame, frame_time=t)                if faces:                    best_face = faces[0]                    face_positions.append(best_face['center_x'])                    print(f"    ✅ Frame {i+1}: Ketemu wajah")                else:                    if face_positions:                        face_positions.append(face_positions[-1])                    else:                        face_positions.append(width // 2)                    print(f"    ⚠️ Frame {i+1}: Enggak ada wajah, pake fallback")            except Exception as e:                if face_positions:                    face_positions.append(face_positions[-1])                else:                    face_positions.append(width // 2)        if face_positions:            print("    ⏳ Lagi hitung trajectory...")            positions = [(pos, height // 2) for pos in face_positions]            smoothed_positions = self.smooth_trajectory(positions, window_size=3)            center_x = int(np.median([pos[0] for pos in smoothed_positions]))            print(f"    ✅ Center optimal: {center_x}")        else:            center_x = width // 2            print("    ⚠️  Pake center crop aja")        center_x = max(target_width // 2, min(width - target_width // 2, center_x))        left = center_x - target_width // 2                print(f"    ⏳ Lagi crop ke {target_width}x{height}")        self.face_cache = {}                cropped_clip = clip.crop(x1=left, width=target_width)        print(f"    ✅ Crop selesai")        return cropped_clip    def close(self):        """Releases resources."""        self.face_cache = {}        print("🎯 Face tracking resources udah di-release")print("   ✅ FaceTracker class udah siap!")# CaptionMaker Classclass CaptionMaker:    """Creates and adds word-by-word captions to video clips."""        def __init__(self, selected_style='clean_white'):        self.font_paths = self.find_available_fonts()        self.selected_style = selected_style        self.styles = CAPTION_STYLES        self.highlight_keywords = HIGHLIGHT_KEYWORDS    def find_available_fonts(self):        """Finds available fonts on the system."""        font_collections = {            'bold': [                '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',                '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',                '/System/Library/Fonts/Helvetica.ttc',                '/Windows/Fonts/arialbd.ttf',            ],            'regular': [                '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',                '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',                '/System/Library/Fonts/Helvetica.ttc',                '/Windows/Fonts/arial.ttf',            ]        }        found_fonts = {'bold': None, 'regular': None}        for font_type, paths in font_collections.items():            for path in paths:                if Path(path).exists():                    found_fonts[font_type] = path                    print(f"    📝 Ketemu font {font_type}: {Path(path).name}")                    break        return found_fonts    def get_font(self, font_type, font_size):        """Gets a font object."""        try:            font_path = self.font_paths.get(font_type)            if font_path:                return ImageFont.truetype(font_path, font_size)            else:                return ImageFont.load_default()        except:            return ImageFont.load_default()    def create_word_image(self, word, video_size, font_size, is_highlighted=False):        """Creates an image of a single word with 3D emboss effect (no background)."""        width, height = video_size        style_config = self.styles.get(self.selected_style, self.styles['clean_white'])        font_type = style_config['font_type']        font = self.get_font(font_type, font_size)        img = Image.new('RGBA', (width, height), (0, 0, 0, 0))        draw = ImageDraw.Draw(img)        temp_bbox = draw.textbbox((0, 0), word, font=font)        text_width = temp_bbox[2] - temp_bbox[0]        text_height = temp_bbox[3] - temp_bbox[1]        # Center horizontally        x = (width - text_width) // 2                # Position at bottom with good margin for 9:16 videos        safe_bottom_margin = int(height * 0.12)        y = height - text_height - safe_bottom_margin        # Make sure text doesn't go too high        if y < height * 0.75:            y = int(height * 0.75)        if is_highlighted:            text_color = (255, 255, 0, 255)        else:            text_color = style_config['text_color']        # Create 3D emboss effect (like in the image)        # Calculate shadow color (darker version of text color)        r, g, b = text_color[0], text_color[1], text_color[2]                # Create darker shadow for emboss effect (subtle, not too dark)        shadow_color = (            max(0, int(r * 0.4)),  # Darker but not too dark            max(0, int(g * 0.4)),            max(0, int(b * 0.4)),            200  # Slightly transparent for subtle effect        )                # Draw shadow/outline layers for 3D emboss effect        # Multiple offset layers create depth        shadow_offsets = [            (-2, -2), (-1, -2), (0, -2), (1, -2), (2, -2),  # Top shadow            (-2, -1), (2, -1),  # Top corners            (-2, 0), (2, 0),  # Side shadows            (-2, 1), (2, 1),  # Bottom corners            (-2, 2), (-1, 2), (0, 2), (1, 2), (2, 2),  # Bottom shadow        ]                # Draw shadow layers (darker, behind)        for offset_x, offset_y in shadow_offsets:            draw.text(                (x + offset_x, y + offset_y),                 word,                 font=font,                 fill=shadow_color            )                # Draw main text (bright, on top) - creates emboss/3D effect        draw.text((x, y), word, font=font, fill=text_color)        return np.array(img)    def add_captions(self, clip, words, clip_start_time):        """Adds word-by-word captions to a video clip."""        if not words:            return clip        caption_clips = []        clip_words = []        for word in words:            word_start = word['start']            word_end = word['end']            word_text = word['word'].strip()            relative_start = word_start - clip_start_time            relative_end = word_end - clip_start_time            if relative_end <= 0 or relative_start >= clip.duration:                continue            relative_start = max(0, relative_start)            relative_end = min(clip.duration, relative_end)            duration = relative_end - relative_start            if duration <= 0.05:                continue            clip_words.append({                'word': word_text.upper(),                'start': relative_start,                'end': relative_end,                'duration': duration            })        if not clip_words:            return clip        video_width, video_height = clip.size        # Font size increased by 20% from previous        # Optimized for 9:16 vertical videos (YouTube Shorts)        base_font_size = max(22, int(video_height * 0.024))  # 20% larger (0.02 * 1.2 = 0.024)        for word_data in clip_words:            word_text = word_data['word']            start_time = word_data['start']            duration = word_data['duration']            is_highlighted = any(keyword.upper() in word_text for keyword in self.highlight_keywords)            word_img = self.create_word_image(word_text, (video_width, video_height), base_font_size, is_highlighted)            word_clip = ImageClip(word_img, duration=duration, transparent=True)            word_clip = word_clip.set_start(start_time).fadein(0.1).fadeout(0.1)            caption_clips.append(word_clip)        if caption_clips:            return CompositeVideoClip([clip] + caption_clips)        return clipprint("   ✅ CaptionMaker class udah siap!")print("\n✅ Semua class sama function udah siap nih!")print("📌 Lanjutin ke cell berikutnya buat proses videonya!\n")

In [ ]:
print("🎬 Auto Clipper By MANG OTHIB - Input Parameters")print("=" * 60)print("\n📝 Masukin detail videonya dulu ya...\n")# YouTube URLyoutube_url = input("🔗 YouTube URL: ").strip()if not youtube_url:    raise ValueError("YouTube URL is required!")# Number of clipsnum_clips_input = input("🎯 Number of clips (default: 3): ").strip()num_clips = int(num_clips_input) if num_clips_input else 3# Minimum durationmin_duration_input = input("⏱️  Min duration in seconds (default: 30): ").strip()min_duration = int(min_duration_input) if min_duration_input else 30# Maximum durationmax_duration_input = input("⏱️  Max duration in seconds (default: 60): ").strip()max_duration = int(max_duration_input) if max_duration_input else 60# Caption styleprint("\n🎨 Pilih style caption lo:")for idx, (style_name, style_config) in enumerate(CAPTION_STYLES.items(), 1):    print(f"   {idx}. {style_name} - {style_config.get('description', '')}")style_choice_input = input(f"\n🎨 Pilih style caption (1-{len(CAPTION_STYLES)}, default: 1): ").strip()style_choice = int(style_choice_input) if style_choice_input else 1style_names = list(CAPTION_STYLES.keys())if 1 <= style_choice <= len(style_names):    selected_style = style_names[style_choice - 1]else:    print("⚠️  Pilihan lo salah, pake default (Clean White) aja ya")    selected_style = "clean_white"print(f"\n✅ Style yang lo pilih: {selected_style}")# Sentiment selectionprint("\n🎭 Pilih sentimen klip yang diinginkan:")print("   1. Lucu/Humor - Momen yang menghibur dan lucu")print("   2. Motivasi - Kata-kata yang memotivasi dan menginspirasi")print("   3. Edukasi - Konten edukatif dan pembelajaran")print("   4. Dramatis - Momen dramatis dan intens")print("   5. Emosional - Momen yang menyentuh hati")print("   6. Inspirasi - Cerita inspiratif dan menggerakkan")print("   7. Kontroversial - Opini kuat atau perdebatan")print("   8. Tutorial - Langkah-langkah atau how-to")print("   9. Storytelling - Cerita menarik")print("   10. Custom - Masukkan sentimen sendiri")print("   0. Viral umum (tanpa sentimen spesifik)")sentiment_choice = input("\n🎭 Pilih sentimen (0-10): ").strip()sentiment_map = {    "1": "lucu",    "2": "motivasi",     "3": "edukasi",    "4": "dramatis",    "5": "emosional",    "6": "inspirasi",    "7": "kontroversial",    "8": "tutorial",    "9": "storytelling"}if sentiment_choice == "10":    sentiment = input("   Masukkan sentimen custom: ").strip()elif sentiment_choice in sentiment_map:    sentiment = sentiment_map[sentiment_choice]else:    sentiment = Noneif sentiment:    print(f"✅ Sentimen terpilih: {sentiment}")else:    print("✅ Mode: Viral umum (tanpa sentimen spesifik)")# Subtitle optionprint("\n📝 Pilihan Subtitle:")print("   1. Ya - Tambahkan subtitle ke video")print("   0. Tidak - Tanpa subtitle")use_subtitle_input = input("\n📝 Gunakan subtitle? (1=Ya, 0=Tidak, default: 1): ").strip()use_subtitle = use_subtitle_input != "0" if use_subtitle_input else Trueif use_subtitle:    print("✅ Subtitle: Aktif")else:    print("✅ Subtitle: Nonaktif")print("\n" + "=" * 60)print("📌 Parameter lo udah disimpan!")print("=" * 60)print(f"\n📝 Ringkasan:")print(f"   URL: {youtube_url}")print(f"   Jumlah clips: {num_clips}")print(f"   Durasi: {min_duration}-{max_duration} detik")print(f"   Style: {selected_style}")print(f"   Sentimen: {sentiment if sentiment else 'Viral umum'}")print(f"   Subtitle: {'Aktif' if use_subtitle else 'Nonaktif'}")print("\n📌 Lanjutin ke cell berikutnya buat download videonya!\n")

In [ ]:
print("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print("\n📥 STEP 1: Download Video dari YouTube")print("=" * 60)try:    downloader = YouTubeDownloader()    video_path, title, duration = downloader.download(youtube_url)        print("\n✅ Download selesai!")    print(f"   Judul: {title}")    print(f"   Durasi: {duration} detik ({duration//60}m {duration%60}s)")    print(f"   Path: {video_path}")        print("\n📌 Lanjutin ke cell berikutnya buat transcribe videonya!\n")    except Exception as e:    print(f"\n❌ Error download video: {e}")    print("💡 Tips:")    print("   - Cek URL YouTube lo")    print("   - Coba video lain")    print("   - Pastikan videonya enggak private atau restricted")    raise

In [ ]:
print("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print("\n🎵 STEP 2: Transcribe Audio pake Whisper AI")print("=" * 60)try:    transcriber = WhisperSingleton()    words, transcript, segments, detected_language = transcriber.transcribe(video_path)        # Store detected language for caption use    video_language = detected_language if detected_language else 'auto'    language_display = transcriber.language_name if hasattr(transcriber, 'language_name') else 'Auto-detect'        if not segments:        print("\n⚠️ Warning: Transcribe gagal atau enggak ada suara yang terdeteksi")        print("💡 Bakal pake random clips aja, enggak pake AI selection")    else:        print(f"\n✅ Transcribe selesai!")        print(f"   Jumlah kata: {len(words)}")        print(f"   Jumlah segment: {len(segments)}")        print(f"   Panjang transcript: {len(transcript)} karakter")        print(f"   🌐 Bahasa yang terdeteksi: {language_display}")        print("\n📌 Lanjutin ke cell berikutnya buat pilih viral clips pake AI!\n")    except Exception as e:    print(f"\n❌ Error pas transcribe: {e}")    print("💡 Lanjutin aja dengan transcript kosong...")    words, transcript, segments = [], "", []    detected_language = None    video_language = 'auto'    language_display = 'Unknown'# Ensure language variables are available globally for next cellsif 'language_display' not in globals():    language_display = 'Auto-detect'if 'detected_language' not in globals():    detected_language = None

In [ ]:
print("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print("\n🧠 STEP 3: AI Pilih Viral Clips Terbaik")print("=" * 60)try:    if not segments:        print("\n⚠️ Enggak ada transcript, pake random clips aja...")        clip_specs = generate_random_clips(duration, num_clips, min_duration, max_duration)        print(f"✅ Udah generate {len(clip_specs)} random clips")    else:        print("\n🤖 Lagi pake Gemini AI buat pilih momen paling viral...")                # Check if API key is valid        if not GEMINI_API_KEY or "YOUR_API_KEY" in GEMINI_API_KEY:            print("⚠️  API key Gemini lo enggak valid nih")            print("💡 Dapatin API key gratis di: https://aistudio.google.com/app/apikey")            raise ValueError("Invalid API key")                ai_selector = GeminiSelector(api_key=GEMINI_API_KEY)                # Show sentiment if provided        if sentiment:            print(f"🎭 Sentimen: {sentiment}")                # Get detected language for AI prompt        ai_language = detected_language if 'detected_language' in globals() and detected_language else 'id'        if ai_language:            print(f"🌐 Pake {language_display if 'language_display' in globals() else ai_language} buat AI selection")                clip_specs = ai_selector.select_clips(            segments, duration, num_clips, min_duration, max_duration,             sentiment=sentiment, language=ai_language        )                # Validate clip durations        valid_clips = []        for clip in clip_specs:            if clip['duration'] >= min_duration and clip['duration'] <= max_duration:                valid_clips.append(clip)            else:                print(f"⚠️  Skip clip yang enggak valid: {clip['duration']:.1f}s (enggak dalam range {min_duration}-{max_duration}s)")                if valid_clips:            clip_specs = valid_clips            sentiment_info = f" (Sentimen: {sentiment})" if sentiment else ""            print(f"\n✅ AI udah pilih {len(clip_specs)} viral clips yang valid{sentiment_info}:")            for i, clip in enumerate(clip_specs, 1):                print(f"   {i}. {clip['title']}")                print(f"      Score: {clip['virality_score']}/100")                print(f"      Waktu: {clip['start']:.1f}s - {clip['end']:.1f}s ({clip['duration']:.1f}s)")        else:            print("⚠️  Enggak ada clips valid dari AI, pake random clips aja...")            clip_specs = generate_random_clips(duration, num_clips, min_duration, max_duration)        if not clip_specs:        raise ValueError("Enggak bisa pilih clips nih")        print("\n📌 Lanjutin ke cell berikutnya buat proses sama export clips!\n")    except Exception as e:    error_msg = str(e)        # Check if it's a quota error    if "429" in error_msg or "quota" in error_msg.lower():        print(f"\n⚠️  Gemini API Quota lo abis nih")        print(f"💡 Solusinya:")        print(f"   1. Dapatin API key baru gratis di: https://aistudio.google.com/app/apikey")        print(f"   2. Tunggu quota reset (biasanya 24 jam)")        print(f"   3. Cek usage lo di: https://ai.google.dev/gemini-api/docs/rate-limits")    else:        print(f"\n❌ Error pas pilih clips: {e}")        print("\n💡 Pake random clips sebagai fallback...")    clip_specs = generate_random_clips(duration, num_clips, min_duration, max_duration)        if clip_specs:        print(f"✅ Udah generate {len(clip_specs)} random clips:")        for i, clip in enumerate(clip_specs, 1):            print(f"   {i}. {clip['title']}")            print(f"      Durasi: {clip['duration']:.1f}s")    else:        print("❌ Gagal generate clips")

In [ ]:
print("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print("\n🎬 STEP 4: Proses & Export Viral Clips")print("=" * 60)# Initialize processorsface_tracker = FaceTracker()caption_maker = CaptionMaker(selected_style)# Initialize SEO generatorseo_generator = Nonetry:    if GEMINI_API_KEY and "YOUR_API_KEY" not in GEMINI_API_KEY:        seo_generator = SEOContentGenerator(api_key=GEMINI_API_KEY)        print("✅ SEO Content Generator udah siap!")    else:        print("⚠️  SEO generation dimatiin (enggak ada API key)")except Exception as e:    print(f"⚠️  SEO generator gagal: {e}")output_files = []print(f"\n🎬 Lagi proses {len(clip_specs)} viral clips...\n")for i, clip_info in enumerate(clip_specs, 1):    start = clip_info['start']    end = clip_info['end']    title_text = clip_info.get('title', f'Clip {i}')    virality_score = clip_info.get('virality_score', 0)    print(f"📹 Clip {i}/{len(clip_specs)}: {title_text}")    print(f"    ⭐ Virality Score: {virality_score}/100")    print(f"    ⏱️  Time: {start:.1f}s to {end:.1f}s")    print(f"    🎨 Caption Style: {CAPTION_STYLES[selected_style]['name']}")    try:        with VideoFileClip(str(video_path)) as video:            # Extract clip            clip = video.subclip(start, end)            # Face tracking and cropping to 9:16 (YouTube Shorts)            print(f"    🎯 Lagi apply face tracking & crop ke 9:16...")            clip = face_tracker.track_and_crop(clip)                        # Verify aspect ratio            final_width, final_height = clip.size            aspect_ratio = final_width / final_height            print(f"    ✅ Ukuran video: {final_width}x{final_height} (Aspect: {aspect_ratio:.2f}, Target: 0.56 untuk 9:16)")            # Add captions (only if subtitle is enabled)            if use_subtitle and words:                # Get language info for display                current_lang = language_display if 'language_display' in globals() else 'detected language'                print(f"    📝 Lagi tambahin captions kata-per-kata ({current_lang})...")                clip = caption_maker.add_captions(clip, words, start)                print(f"    ✅ Captions udah ditambahin dalam {current_lang}")            elif not use_subtitle:                print(f"    ⏩ Skip captions (subtitle dimatiin)")            elif not words:                print(f"    ⏩ Skip captions (enggak ada transcript)")            # Export - Rename file based on title            # Sanitize title for filename            safe_title = title_text.replace(' ', '_').replace('/', '_').replace('\\', '_')            safe_title = ''.join(c for c in safe_title if c.isalnum() or c in ('_', '-', '.'))[:50]  # Limit length            filename = f"{safe_title}_{virality_score}pts.mp4"            output_path = OUTPUT_DIR / filename            print(f"    🎥 Lagi encode video dengan kualitas HD maksimal...")            print(f"    💡 Ini mungkin agak lama, tapi kualitasnya mantap!")                        # Buat progress bar untuk encoding            # MoviePy encoding biasanya 2-3x lebih lama dari durasi video            estimated_time = clip.duration * 2.5            encoding_pbar = tqdm(                total=100,                desc=f"    🎬 Encoding Clip {i}",                bar_format='{l_bar}{bar}| {n_fmt}% [{elapsed}<{remaining}]',                unit='%',                initial=0            )                        # Update progress bar secara periodik            import threading            import time                        stop_progress = threading.Event()                        def update_progress():                start_time = time.time()                while not stop_progress.is_set():                    elapsed = time.time() - start_time                    if estimated_time > 0:                        percent = min(99, int((elapsed / estimated_time) * 100))                        encoding_pbar.n = percent                        encoding_pbar.refresh()                    time.sleep(0.5)                        progress_thread = threading.Thread(target=update_progress, daemon=True)            progress_thread.start()                        try:                clip.write_videofile(                    str(output_path),                    codec='libx264',                    audio_codec='aac',                    preset='slow',  # Best quality preset (slower but much better)                    bitrate='12000k',  # Very high bitrate for maximum HD                    audio_bitrate='320k',  # High quality audio                    ffmpeg_params=[                        '-crf', '15',  # Lower CRF = better quality (15 is near-lossless)                        '-pix_fmt', 'yuv420p',                        '-profile:v', 'high',  # H.264 high profile for better quality                        '-level', '4.2',  # H.264 level for HD support                        '-movflags', '+faststart'  # Better streaming/playback                    ],                    verbose=False,                    logger=None,                    temp_audiofile=str(TEMP_DIR / f'temp_audio_{i}.m4a'),                    remove_temp=True                )            finally:                stop_progress.set()                encoding_pbar.n = 100                encoding_pbar.refresh()                encoding_pbar.close()                        # Show file info            file_size_mb = os.path.getsize(output_path) / (1024 * 1024)            print(f"    ✅ Tersimpan: {filename}")            print(f"    📊 Resolusi: {final_width}x{final_height} | Ukuran: {file_size_mb:.2f} MB | Kualitas: CRF 15 (Near-Lossless)")                        # Extract transcript for this clip            clip_transcript = ""            if segments:                clip_segments = [seg for seg in segments if seg['start'] >= start and seg['end'] <= end]                clip_transcript = " ".join([seg['text'] for seg in clip_segments])                        # Generate SEO content            seo_content = {                'title': title_text,                'description': '',                'hashtags': [],                'tags': []            }                        if seo_generator and clip_transcript:                print(f"    🔍 Lagi generate konten SEO-optimized...")                try:                    ai_language = detected_language if 'detected_language' in globals() and detected_language else 'id'                    seo_content = seo_generator.generate_seo_content(                        title_text,                         clip_transcript,                         language=ai_language,                        platform='youtube'                    )                    print(f"    ✅ Konten SEO udah di-generate!")                except Exception as e:                    print(f"    ⚠️  Generate SEO gagal: {e}")                    # Use fallback                    seo_content['title'] = title_text                    seo_content['description'] = f"{title_text}. Watch this viral clip! 🔥"                    seo_content['hashtags'] = ['#viral', '#shorts', '#trending']                    seo_content['tags'] = ['viral', 'shorts', 'trending']            else:                # Fallback SEO content                seo_content['title'] = title_text                seo_content['description'] = f"{title_text}. Watch this viral clip! 🔥"                seo_content['hashtags'] = ['#viral', '#shorts', '#trending']                seo_content['tags'] = ['viral', 'shorts', 'trending']                        # Store file info with SEO content            output_files.append({                'title': title_text,                'filename': filename,                'path': str(output_path),                'size_mb': file_size_mb,                'virality_score': virality_score,                'duration': f"{start:.1f}s - {end:.1f}s",                'seo_title': seo_content['title'],                'seo_description': seo_content['description'],                'seo_hashtags': seo_content['hashtags'],                'seo_tags': seo_content['tags']            })    except Exception as e:        print(f"    ❌ Error proses clip {i}: {e}")        continue# Cleanupface_tracker.close()cleanup_temp_files()print("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print(f"\n✅ PROSES SELESAI!")print("=" * 60)print(f"\n📹 Udah generate {len(output_files)} clips:")for i, file_info in enumerate(output_files, 1):    if isinstance(file_info, dict):        print(f"   {i}. {file_info['title']} ({file_info['size_mb']:.2f} MB)")    else:        # Backward compatibility        file_size = Path(file_info).stat().st_size / (1024 * 1024)        print(f"   {i}. {Path(file_info).name} ({file_size:.2f} MB)")print("\n📌 Lanjutin ke cell berikutnya buat liat download links!\n")

In [ ]:
from IPython.display import HTML, displayfrom google.colab import filestry:    import ipywidgets as widgets    from IPython.display import clear_output    WIDGETS_AVAILABLE = Trueexcept:    WIDGETS_AVAILABLE = Falseprint("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB ✨")print("=" * 60)print("\n📥 STEP 5: Konten SEO-Optimized & Download Links")print("=" * 60)if not output_files:    print("\n⚠️  Enggak ada file output yang di-generate")    print("💡 Cek error di atas dulu, terus coba lagi ya")else:    print(f"\n✅ {len(output_files)} video clips udah siap dengan konten SEO-optimized!")    print("\n" + "=" * 60)        # Display SEO-optimized content for each clip    print("\n" + "=" * 60)    print("✨ Auto Clipper By MANG OTHIB ✨")    print("=" * 60)    print("\n📋 SEO-OPTIMIZED CONTENT FOR EACH CLIP")    print("=" * 60)        for i, file_info in enumerate(output_files, 1):        if isinstance(file_info, dict):            seo_title = file_info.get('seo_title', file_info.get('title', ''))            seo_description = file_info.get('seo_description', '')            seo_hashtags = file_info.get('seo_hashtags', [])            seo_tags = file_info.get('seo_tags', [])            file_path = file_info['path']            filename = file_info['filename']            size_mb = file_info.get('size_mb', 0)            virality_score = file_info.get('virality_score', 0)                        print(f"\n{'='*60}")            print(f"📹 CLIP {i}: {file_info.get('title', 'Untitled')}")            print(f"{'='*60}")            print(f"\n📝 SEO TITLE:")            print(f"   {seo_title}")            print(f"\n📄 SEO DESCRIPTION:")            print(f"   {seo_description}")            print(f"\n🏷️  HASHTAGS ({len(seo_hashtags)}):")            hashtags_str = " ".join(seo_hashtags) if isinstance(seo_hashtags, list) else seo_hashtags            print(f"   {hashtags_str}")            print(f"\n🔖 TAGS ({len(seo_tags) if isinstance(seo_tags, list) else 0}):")            tags_str = ", ".join(seo_tags) if isinstance(seo_tags, list) else seo_tags            print(f"   {tags_str}")            print(f"\n⬇️  DOWNLOAD LINK:")            print(f"   Path: {file_path}")            print(f"   Filename: {filename}")            print(f"   Size: {size_mb:.2f} MB")            print(f"   Score: {virality_score}/100")        # Create download buttons using IPython widgets    if WIDGETS_AVAILABLE:        print("\n" + "=" * 60)        print("✨ Auto Clipper By MANG OTHIB ✨")        print("=" * 60)        print("\n🎬 QUICK DOWNLOAD BUTTONS")        print("=" * 60)        print("\nKlik tombol di bawah buat download:\n")                for i, file_info in enumerate(output_files, 1):            if isinstance(file_info, dict):                title = file_info.get('title', 'Untitled')                file_path = file_info['path']                filename = file_info['filename']                size_mb = file_info.get('size_mb', 0)                                # Create download function with proper closure                def make_download_func(path, idx):                    def download_func(btn):                        print(f"⏳ Lagi download clip {idx}...")                        files.download(path)                        print(f"✅ Download udah dimulai untuk clip {idx}!")                    return download_func                                # Create button                download_btn = widgets.Button(                    description=f'⬇️ Download {i}',                    button_style='primary',                    layout=widgets.Layout(width='180px', height='40px')                )                download_btn.on_click(make_download_func(file_path, i))                                # Create info display                info_text = widgets.HTML(f"""                <div style="padding: 12px; margin: 5px 0; background: #f9f9f9; border-radius: 5px; border-left: 4px solid #1a73e8;">                    <b style="color: #1a73e8; font-size: 14px;">Clip {i}: {title[:50]}</b><br>                    <span style="color: #666; font-size: 11px;">📁 {filename} | 📊 {size_mb:.2f} MB</span>                </div>                """)                                # Display                display(widgets.HBox([info_text, download_btn]))            else:                # Fallback for backward compatibility                file_path = file_info                filename = Path(file_path).name                file_size = Path(file_path).stat().st_size / (1024 * 1024)                                def make_download_func(path, idx):                    def download_func(btn):                        print(f"⏳ Lagi download clip {idx}...")                        files.download(path)                        print(f"✅ Download udah dimulai untuk clip {idx}!")                    return download_func                                download_btn = widgets.Button(                    description=f'⬇️ Download',                    button_style='primary',                    layout=widgets.Layout(width='150px', height='35px')                )                download_btn.on_click(make_download_func(file_path, i))                                info_text = widgets.HTML(f"""                <div style="padding: 10px; margin: 5px 0; background: #f9f9f9; border-radius: 5px;">                    <b style="color: #1a73e8; font-size: 14px;">{i}. {filename}</b><br>                    <span style="color: #666; font-size: 12px;">📊 Size: {file_size:.2f} MB</span>                </div>                """)                                display(widgets.HBox([info_text, download_btn]))    else:        # Fallback: Display HTML with SEO content        html_parts = []        html_parts.append("""        <div style="font-family: Arial, sans-serif; padding: 20px; background: #f5f5f5; border-radius: 10px; margin: 20px 0;">            <h2 style="color: #333; margin-bottom: 20px;">🎬 Auto Clipper By MANG OTHIB - SEO Content</h2>        """)                for i, file_info in enumerate(output_files, 1):            if isinstance(file_info, dict):                title = file_info.get('title', 'Untitled')                seo_title = file_info.get('seo_title', title)                seo_description = file_info.get('seo_description', '')                seo_hashtags = file_info.get('seo_hashtags', [])                seo_tags = file_info.get('seo_tags', [])                filename = file_info['filename']                file_path = file_info['path']                size_mb = file_info.get('size_mb', 0)                virality_score = file_info.get('virality_score', 0)                                hashtags_str = " ".join(seo_hashtags) if isinstance(seo_hashtags, list) else str(seo_hashtags)                tags_str = ", ".join(seo_tags) if isinstance(seo_tags, list) else str(seo_tags)                                html_parts.append(f"""            <div style="background: white; padding: 20px; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">                <h3 style="margin: 0 0 15px 0; color: #1a73e8; font-size: 18px; border-bottom: 2px solid #1a73e8; padding-bottom: 10px;">Clip {i}: {title}</h3>                                <div style="margin-bottom: 15px;">                    <strong style="color: #333; font-size: 14px;">📝 SEO Title:</strong>                    <p style="margin: 5px 0; color: #555; font-size: 14px; font-weight: bold;">{seo_title}</p>                </div>                                <div style="margin-bottom: 15px;">                    <strong style="color: #333; font-size: 14px;">📄 SEO Description:</strong>                    <p style="margin: 5px 0; color: #666; font-size: 13px; line-height: 1.6;">{seo_description}</p>                </div>                                <div style="margin-bottom: 15px;">                    <strong style="color: #333; font-size: 14px;">🏷️ Hashtags:</strong>                    <p style="margin: 5px 0; color: #1a73e8; font-size: 13px;">{hashtags_str}</p>                </div>                                <div style="margin-bottom: 15px;">                    <strong style="color: #333; font-size: 14px;">🔖 Tags:</strong>                    <p style="margin: 5px 0; color: #666; font-size: 12px; font-family: monospace;">{tags_str}</p>                </div>                                <div style="margin-top: 15px; padding-top: 15px; border-top: 1px solid #ddd;">                    <strong style="color: #333; font-size: 14px;">⬇️ Download:</strong>                    <p style="margin: 5px 0; color: #999; font-size: 12px; font-family: monospace;">📁 {filename} | 📊 {size_mb:.2f} MB | ⭐ {virality_score}/100</p>                    <p style="margin: 5px 0; color: #666; font-size: 11px;">Path: {file_path}</p>                </div>            </div>            """)            else:                file_path = file_info                filename = Path(file_path).name                file_size = Path(file_path).stat().st_size / (1024 * 1024)                                html_parts.append(f"""            <div style="background: white; padding: 15px; margin-bottom: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">                <h3 style="margin: 0 0 8px 0; color: #1a73e8; font-size: 16px;">{i}. {filename}</h3>                <p style="margin: 5px 0; color: #666; font-size: 13px;">📊 Size: {file_size:.2f} MB</p>            </div>            """)                html_parts.append("</div>")        display(HTML(''.join(html_parts)))        # Create Python download functions for each file    print("\n" + "=" * 60)    print("✨ Auto Clipper By MANG OTHIB ✨")    print("=" * 60)    print("\n💡 CARA PAKE KONTEN SEO-OPTIMIZED LO")    print("=" * 60)    print("\n✅ Setiap clip udah include:")    print("   📝 SEO-optimized Title (buat YouTube/TikTok/Instagram)")    print("   📄 SEO-optimized Description (150-200 kata)")    print("   🏷️  Hashtags (10-15 trending + niche hashtags)")    print("   🔖 Tags (15-20 SEO keywords)")    print("   ⬇️  Download link buat videonya")    print("\n📌 Copy konten SEO di atas buat setiap clip pas upload ya!")    print("\n" + "=" * 60)    print("💡 CARA DOWNLOAD")    print("=" * 60)    print("\n📌 Cara 1: Klik tombol 'Download' di atas setiap clip")    print("📌 Cara 2: Pake code di bawah buat download programmatically")    print("\n💻 Python Download Code (copy terus jalanin di cell baru):")    print("\n```python")    print("from google.colab import files")    print()    for i, file_info in enumerate(output_files, 1):        if isinstance(file_info, dict):            print(f"# {i}. {file_info.get('title', 'Untitled')}")            print(f"files.download('{file_info['path']}')  # Downloads as: {file_info['filename']}")        else:            print(f"# {i}. {Path(file_info).name}")            print(f"files.download('{file_info}')")        if i < len(output_files):            print()    print("```")    print("\n💻 Cara 3: Download via file browser:")    print("   1. Klik icon folder (📁) di sidebar kiri")    print("   2. Navigate ke: output/")    print("   3. Right-click file → Download")    print("\n📌 File-file otomatis di-rename sesuai judul clip lo!")    print("\n💡 TIP: Copy konten SEO (title, description, hashtags, tags) di atas")    print("   terus paste pas upload ke YouTube Shorts, TikTok, atau Instagram Reels!")        print("\n" + "=" * 60)    print("🎉 SELESAI! Clips lo udah siap!")    print("=" * 60)    print("\n✨ Auto Clipper By MANG OTHIB ✨")    print("=" * 60)        print(f"\n✅ Lo bisa upload clips ini ke:")    print(f"   • TikTok")    print(f"   • Instagram Reels")    print(f"   • YouTube Shorts")        print(f"\n🚀 Siap go viral! 🎬\n")print("\n" + "=" * 60)print("✨ Auto Clipper By MANG OTHIB - Process Complete!")print("=" * 60)